###Dimension Market table creation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import requests
import boto3

In [0]:
%run /Workspace/Agmarknet/setup/utilities

In [0]:
dbutils.widgets.text('catalog','agmarknet')


In [0]:
catalog = dbutils.widgets.get('catalog')
print(catalog)

In [0]:
print(silver_schema)

In [0]:
access_key_id = dbutils.secrets.get(scope="aws",key="access_key_id")
access_key = dbutils.secrets.get(scope="aws",key="access_key")

In [0]:
###Read the api and save the json file to S3 bucket
import requests
import boto3

url = "https://api.agmarknet.gov.in/v1/market-district-state"

headers = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": "https://agmarknet.gov.in",
    "Referer": "https://agmarknet.gov.in/",
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
s3 = boto3.client("s3", aws_access_key_id=access_key_id, aws_secret_access_key=access_key)
s3.put_object(
    Bucket="agmarknet-pc",
    Key=f"agmarknet_market_district_state.json",
    Body=response.content
)

print(response.status_code)
print(response.headers.get("Content-Type"))
print(response.text[:1000])

In [0]:
market_dist_st = f's3://agmarknet-pc/agmarknet_market_district_state.json'
print(market_dist_st)

In [0]:
#read json file containing markets district state data from s3
df_all = spark.read.option("multiline", "true").json(market_dist_st)
df_all.count()

In [0]:
df_all.printSchema()

In [0]:
display(df_all)

####writing to bronze schema

In [0]:
df_all.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed","true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{bronze_schema}.dim_market')

%md
####writing to silver schema

In [0]:
df_all.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed","true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{silver_schema}.dim_market')

%md
####writing to gold schema

In [0]:
df_all.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed","true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{gold_schema}.dim_market')